# GateGuard — Rule-based Event Pipeline v8

**v8 변경사항**
- Fix 1: GT 라벨을 파일명이 아닌 **상위 폴더명** 기준으로 수정
- Fix 2: `run_pipeline()` 파라미터 하드코딩 제거 (gate_conf / near_gate_margin / jump_multiplier 등)
- Fix 3: `CONFIG_DICT_V2`에 crawl/tailgating 파라미터 전부 반영
- Fix 4: verbose_rules=True 시 rule별 탈락 사유 + 진단 로그 출력
- EfficientNet 2차 검증 (v7 유지)

In [ ]:
!pip install -q ultralytics supervision opencv-python-headless tqdm

## 1. Drive 마운트 & 경로 설정

In [ ]:
import os, shutil, json
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/ㅈㅎㅊ'

VIDEO_FOLDERS = [
    f'{DRIVE_ROOT}/변환 완료',
    f'{DRIVE_ROOT}/직접 촬영',
]

GATE_MODEL_PATH   = f'{DRIVE_ROOT}/gate_best.pt'
FACE_MODEL_PATH   = f'{DRIVE_ROOT}/yolov11n-face.pt'
OUTPUT_DIR        = '/content/pipeline_output'
DRIVE_SAVE        = f'{DRIVE_ROOT}/pipeline_output'
CLIPS_DIR         = f'{OUTPUT_DIR}/clips'
CHECKPOINT        = None
EFFICIENTNET_PATH = f"/content/drive/MyDrive/GateGuard_runs/best_model.pth"
GATE_CACHE_FRAMES = 60

# 포함할 하위 폴더명 (영어 이름만)
INCLUDE_SUBFOLDERS  = {'crawling', 'jump', 'nomal', 'normal', 'tailgating'}
MAX_FILES_PER_CLASS = 20

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CLIPS_DIR,  exist_ok=True)

all_videos = []
for parent in VIDEO_FOLDERS:
    if not os.path.exists(parent):
        print(f'[SKIP] 폴더 없음: {parent}')
        continue
    for sub in sorted(os.listdir(parent)):
        sub_path = os.path.join(parent, sub)
        if not os.path.isdir(sub_path) or sub not in INCLUDE_SUBFOLDERS:
            continue
        files = sorted([
            os.path.join(sub_path, f)
            for f in os.listdir(sub_path)
            if f.lower().endswith(('.mp4', '.mov', '.avi', '.mkv'))
        ])
        picked = files[:MAX_FILES_PER_CLASS]
        all_videos.extend(picked)
        print(f'  {os.path.basename(parent)}/{sub}: {len(picked)}개 (전체 {len(files)}개 중)')

print(f'\n합계: {len(all_videos)}개 영상')
print(f'gate model exists: {os.path.exists(GATE_MODEL_PATH)}')

## 2. Drive → 로컬 복사

In [ ]:
from tqdm.notebook import tqdm as tqdm_nb
import time

LOCAL_VIDEO_DIR = '/content/videos'
os.makedirs(LOCAL_VIDEO_DIR, exist_ok=True)

def copy_with_retry(src, dst, retries=3, wait=5):
    for attempt in range(retries):
        try:
            shutil.copy2(src, dst)
            return True
        except OSError as e:
            print(f'\n[WARN] 복사 실패 ({attempt+1}/{retries}): {os.path.basename(src)} — {e}')
            if attempt < retries - 1:
                print(f'  {wait}초 후 재시도...')
                time.sleep(wait)
                # Drive 재마운트 시도
                try:
                    from google.colab import drive
                    drive.mount('/content/drive', force_remount=True)
                except Exception:
                    pass
    print(f'[SKIP] 복사 포기: {os.path.basename(src)}')
    return False

local_videos = []
skipped = []

for src in tqdm_nb(all_videos, desc='Drive → 로컬'):
    rel = os.path.relpath(src, DRIVE_ROOT)
    dst = os.path.join(LOCAL_VIDEO_DIR, rel.replace(os.sep, '_'))
    if not os.path.exists(dst):
        ok = copy_with_retry(src, dst)
        if not ok:
            skipped.append(src)
            continue
    local_videos.append(dst)

print(f'\n복사 완료: {len(local_videos)}개')
if skipped:
    print(f'스킵된 파일 {len(skipped)}개:')
    for s in skipped:
        print(f'  {os.path.basename(s)}')

## 3. 모델 로드

In [ ]:
from ultralytics import YOLO

_person_model = YOLO('yolo11n.pt')
print('Person YOLO: 완료')

_gate_model = None
if os.path.exists(GATE_MODEL_PATH):
    _gate_model = YOLO(GATE_MODEL_PATH)
    print(f'Gate model: 완료 ({GATE_MODEL_PATH})')
else:
    print('[WARN] gate_best.pt 없음 → gate 위치 필터 비활성화, CONFIG_DICT fallback 사용')

_face_model = None
if os.path.exists(FACE_MODEL_PATH):
    _face_model = YOLO(FACE_MODEL_PATH)
    print('Face model: 완료')
else:
    try:
        from huggingface_hub import hf_hub_download
        face_pt = hf_hub_download('arnabdhar/YOLOv8-Face-Detection', 'model.pt', local_dir='/content')
        _face_model = YOLO(face_pt)
        print('Face model: HuggingFace에서 로드')
    except Exception as e:
        print(f'Face model 로드 실패 → 블러 생략: {e}')

In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from pathlib import Path

# ── EfficientNet 설정 ─────────────────────────────────────────────────────
EFF_CLASSES        = ("jump", "crawling", "tailgating", "unpaid", "normal")
EFF_NUM_FRAMES     = 16
EFF_CONF_THRESHOLD = 0.50   # normal 판정 최소 신뢰도 (이 이상이면 FP로 제거)

class _EfficientNetClipClassifier(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        backbone      = models.efficientnet_b0(weights=None)
        self.features = backbone.features
        self.avgpool  = backbone.avgpool
        self.dropout  = nn.Dropout(p=0.3)
        self.fc       = nn.Linear(1280, num_classes)
    def forward(self, clips):
        b, t = clips.shape[:2]
        x = clips.view(b * t, *clips.shape[2:])
        x = self.features(x)
        x = self.avgpool(x).flatten(1)
        x = x.view(b, t, 1280).mean(dim=1)
        x = self.dropout(x)
        return self.fc(x)

_eff_device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
_eff_model  = None

if os.path.exists(EFFICIENTNET_PATH):
    _eff_model = _EfficientNetClipClassifier(num_classes=len(EFF_CLASSES)).to(_eff_device)
    ckpt = torch.load(EFFICIENTNET_PATH, map_location=_eff_device)
    _eff_model.load_state_dict(ckpt["model_state"])
    _eff_model.eval()
    print(f"EfficientNet 로드 완료: {EFFICIENTNET_PATH}")
else:
    print(f"[WARN] EfficientNet 없음 → 2차 검증 비활성화: {EFFICIENTNET_PATH}")

_eff_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def eff_verify(clip_path, threshold=EFF_CONF_THRESHOLD):
    """클립 경로 → EfficientNet 분류 결과 dict (모델 없거나 clip_path=None이면 None)."""
    if _eff_model is None or clip_path is None:
        return None
    cap   = cv2.VideoCapture(clip_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release(); return None
    idxs   = np.linspace(0, max(total - 1, 0), EFF_NUM_FRAMES).astype(int)
    frames = []
    for idx in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, f = cap.read()
        if ok and f is not None:
            frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    cap.release()
    if not frames: return None
    while len(frames) < EFF_NUM_FRAMES:
        frames.append(frames[-1])
    tensors = [_eff_transform(f) for f in frames[:EFF_NUM_FRAMES]]
    clip    = torch.stack(tensors, dim=0).unsqueeze(0).to(_eff_device)
    with torch.no_grad():
        probs = torch.softmax(_eff_model(clip), dim=1).squeeze(0).cpu().tolist()
    pred_idx   = int(max(range(len(EFF_CLASSES)), key=lambda i: probs[i]))
    confidence = probs[pred_idx]
    return {
        "prediction": EFF_CLASSES[pred_idx] if confidence >= threshold else "unknown",
        "confidence":  round(confidence, 4),
        "probs":       {c: round(probs[i], 4) for i, c in enumerate(EFF_CLASSES)},
    }

print(f"EfficientNet 2차 분류기 준비 완료  threshold={EFF_CONF_THRESHOLD}  device={_eff_device}")

## 4. 모듈 정의
### 4-A. YOLODetector / GateDetector / blur_faces

In [ ]:
import numpy as np
import cv2
from dataclasses import dataclass
from typing import List, Optional, Dict, Any

@dataclass
class Detection:
    x1: float; y1: float; x2: float; y2: float; confidence: float
    @property
    def cx(self): return (self.x1 + self.x2) / 2
    @property
    def cy(self): return (self.y1 + self.y2) / 2
    @property
    def width(self): return self.x2 - self.x1
    @property
    def height(self): return self.y2 - self.y1
    @property
    def aspect_ratio(self): return self.height / (self.width + 1e-6)

class YOLODetector:
    def __init__(self, conf_threshold=0.4):
        self.model = _person_model
        self.conf_threshold = conf_threshold
    def detect_persons(self, frame):
        results = self.model(frame, classes=[0], conf=self.conf_threshold, verbose=False)
        out = []
        for r in results:
            if r.boxes is None: continue
            for box in r.boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                out.append(Detection(x1=x1, y1=y1, x2=x2, y2=y2, confidence=float(box.conf[0])))
        return out

class GateDetector:
    def __init__(self, conf_thres=0.25):
        self.model = _gate_model
        self.conf_thres = conf_thres
    @property
    def enabled(self): return self.model is not None
    def detect_best(self, frame):
        if frame is None or self.model is None: return None
        h, w = frame.shape[:2]
        results = self.model.predict(source=frame, conf=self.conf_thres, verbose=False)
        dets = []
        for result in results:
            if result.boxes is None: continue
            for box in result.boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                dets.append({
                    'x1': max(0, min(int(x1), w-1)), 'y1': max(0, min(int(y1), h-1)),
                    'x2': max(0, min(int(x2), w-1)), 'y2': max(0, min(int(y2), h-1)),
                    'conf': round(float(box.conf[0]), 4)
                })
        return max(dets, key=lambda d: d['conf']) if dets else None

def blur_faces(frame):
    if _face_model is None: return frame
    out = frame.copy()
    for r in _face_model(frame, verbose=False):
        if r.boxes is None: continue
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            roi = out[y1:y2, x1:x2]
            if roi.size: out[y1:y2, x1:x2] = cv2.GaussianBlur(roi, (51, 51), 0)
    return out

print('YOLODetector / GateDetector 완료')

### 4-B. JumpDetector (Y축 변위 기반, 기존 검증값 사용)

In [ ]:
from collections import deque

class JumpDetector:
    """
    y2(발 끝) 변위 기반 점프 감지.
    - dynamic_multiplier : threshold = 정상 y2 변위 평균 × multiplier
    - min_score          : score(y2 변위/키) 최소값
    - _gate_min          : threshold 하한. gate 탐지 시 gate_height × 0.08로 자동 갱신
    """
    def __init__(self, window_frames=30, jump_ratio=0.30,
                 calibration_frames=90, dynamic_multiplier=5.0,
                 min_score=0.25):
        self.window_frames      = window_frames
        self.jump_ratio         = jump_ratio
        self.calibration_frames = calibration_frames
        self.dynamic_multiplier = dynamic_multiplier
        self.min_score          = min_score
        self._history           = {}
        self._frame_count       = 0
        self._calib_disps       = []
        self._dynamic_threshold = None
        self._gate_min          = 30.0  # gate 미탐지 시 기본 하한

    def set_gate_height(self, gate_h):
        """gate 탐지 후 gate_height 기반으로 threshold 하한 재계산."""
        self._gate_min = max(gate_h * 0.08, 15.0)
        if self._dynamic_threshold is not None:
            self._dynamic_threshold = max(self._dynamic_threshold, self._gate_min)

    def _upward_disp(self, tid):
        h = self._history.get(tid)
        if not h or len(h) < 5: return 0.0, 0.0
        y2_vals = [y2 for y2, _ in h]
        avg_h   = float(np.mean([ht for _, ht in h]))
        n       = len(y2_vals)
        base    = float(np.mean(y2_vals[:max(1, n // 4)]))
        return base - min(y2_vals), avg_h

    def _try_calibrate(self):
        if not self._calib_disps: return
        mean_disp = float(np.mean(self._calib_disps))
        self._dynamic_threshold = max(mean_disp * self.dynamic_multiplier, self._gate_min)
        print(f'[JumpDetector] 보정 완료: 정상 y2 변위={mean_disp:.1f}px '
              f'→ threshold={self._dynamic_threshold:.1f}px '
              f'(×{self.dynamic_multiplier}, min={self._gate_min:.1f}px)')

    def update(self, tid, xyxy):
        x1, y1, x2, y2 = xyxy
        h = y2 - y1
        if tid not in self._history:
            self._history[tid] = deque(maxlen=self.window_frames)
        self._history[tid].append((y2, h))
        if self._frame_count < self.calibration_frames:
            disp, _ = self._upward_disp(tid)
            if disp > 0: self._calib_disps.append(disp)
        elif self._dynamic_threshold is None:
            self._try_calibrate()
        self._frame_count += 1

    def is_jump(self, tid):
        upward, avg_h = self._upward_disp(tid)
        if avg_h == 0: return False
        score = upward / avg_h
        if score < self.min_score: return False
        if self._dynamic_threshold is not None:
            return upward > self._dynamic_threshold
        return score > self.jump_ratio

    def get_score(self, tid):
        upward, avg_h = self._upward_disp(tid)
        return upward / avg_h if avg_h > 0 else 0.0

    def transfer_history(self, old_tid, new_tid):
        """ID switch 보정: old_tid의 히스토리를 new_tid로 이어받기."""
        if old_tid in self._history:
            self._history[new_tid] = self._history.pop(old_tid)

    def cleanup(self, active_tids):
        for tid in list(self._history.keys()):
            if tid not in active_tids:
                del self._history[tid]

print('JumpDetector 완료 (gate_height 기반 threshold, set_gate_height() 지원)')

### 4-C. GateRelation (gate 기준 상대 좌표, 위치 필터용)

In [ ]:
from dataclasses import dataclass as dc

@dc
class GateRelation:
    in_gate: bool
    rel_x: float
    rel_y: float
    vertical_zone: str  # 'top' | 'middle' | 'bottom'
    overlap_iou: float

def compute_gate_relation(person, gate):
    if gate is None: return None
    gx1, gy1, gx2, gy2 = gate['x1'], gate['y1'], gate['x2'], gate['y2']
    gw = gx2 - gx1; gh = gy2 - gy1
    if gw <= 0 or gh <= 0: return None
    pcx, pcy = person.cx, person.cy
    in_gate = (gx1 <= pcx <= gx2) and (gy1 <= pcy <= gy2)
    rel_x = (pcx - gx1) / gw
    rel_y = (pcy - gy1) / gh
    zone = 'top' if rel_y < 0.33 else ('middle' if rel_y < 0.66 else 'bottom')
    ix1 = max(person.x1, gx1); iy1 = max(person.y1, gy1)
    ix2 = min(person.x2, gx2); iy2 = min(person.y2, gy2)
    inter = max(0, ix2-ix1) * max(0, iy2-iy1)
    union = (person.x2-person.x1)*(person.y2-person.y1) + gw*gh - inter
    iou = inter / union if union > 0 else 0.0
    return GateRelation(in_gate=in_gate, rel_x=rel_x, rel_y=rel_y,
                        vertical_zone=zone, overlap_iou=iou)

def near_gate_x(person, gate, margin_ratio=0.2):
    """person cx가 gate x-range ± margin 안에 있는지. gate 없으면 True."""
    if gate is None: return True
    gx1, gx2 = gate['x1'], gate['x2']
    margin = (gx2 - gx1) * margin_ratio
    return (gx1 - margin) <= person.cx <= (gx2 + margin)

print('GateRelation 완료')

### 4-D. TrackedPerson / EventCandidate / Rules

In [ ]:
from dataclasses import dataclass, field
from collections import defaultdict
from typing import Dict, List, Optional, Tuple

@dataclass
class Zone:
    x1:int; y1:int; x2:int; y2:int
    def contains_point(self, x, y): return self.x1<=x<=self.x2 and self.y1<=y<=self.y2

@dataclass
class PassLine:
    x1:int; y1:int; x2:int; y2:int

@dataclass
class GateZoneConfig:
    camera_id:str; frame_width:int; frame_height:int
    gate_zone:Zone; pass_line:PassLine; jump_zone:Zone; crawl_zone:Zone
    tailgate_time_window_s:float   = 3.0
    tailgate_distance_thresh:float = 200.0
    unpaid_hold_s:float            = 5.0
    jump_min_frames:int            = 3
    crawl_min_frames:int           = 10    # v6: recall 회복
    crawl_aspect_ratio_thresh:float= 1.6   # v6: recall 회복
    tailgating_min_frames:int      = 5     # v6: FP 억제
    gate_overlap_thresh:float      = 0.25  # v6: 더 엄격
    tailgate_max_dist:float        = 200.0 # v6 NEW: 두 사람 거리 상한

    @classmethod
    def from_dict(cls, d):
        return cls(
            camera_id=d['camera_id'], frame_width=d['frame_width'],
            frame_height=d['frame_height'], gate_zone=Zone(**d['gate_zone']),
            pass_line=PassLine(**d['pass_line']), jump_zone=Zone(**d['jump_zone']),
            crawl_zone=Zone(**d['crawl_zone']),
            tailgate_time_window_s   =d.get('tailgate_time_window_s',    3.0),
            tailgate_distance_thresh =d.get('tailgate_distance_thresh',  200.0),
            unpaid_hold_s            =d.get('unpaid_hold_s',             5.0),
            jump_min_frames          =d.get('jump_min_frames',           3),
            crawl_min_frames         =d.get('crawl_min_frames',          10),
            crawl_aspect_ratio_thresh=d.get('crawl_aspect_ratio_thresh', 1.6),
            tailgating_min_frames    =d.get('tailgating_min_frames',     5),
            gate_overlap_thresh      =d.get('gate_overlap_thresh',       0.25),
            tailgate_max_dist        =d.get('tailgate_max_dist',         200.0),
        )

@dataclass
class TrackedPerson:
    track_id:int; x1:float; y1:float; x2:float; y2:float
    @property
    def cx(self): return (self.x1+self.x2)/2
    @property
    def cy(self): return (self.y1+self.y2)/2
    @property
    def aspect_ratio(self): return (self.y2-self.y1)/((self.x2-self.x1)+1e-6)

@dataclass
class EventCandidate:
    event_type:str; start_frame:int; end_frame:int; track_ids:List[int]
    confidence:float; reason:str
    status:str='candidate'
    efficientnet_result:Optional[dict]=None

def _bbox_overlap_ratio(px1, py1, px2, py2, gx1, gy1, gx2, gy2) -> float:
    ix1 = max(px1, gx1);  iy1 = max(py1, gy1)
    ix2 = min(px2, gx2);  iy2 = min(py2, gy2)
    if ix2 <= ix1 or iy2 <= iy1: return 0.0
    inter  = (ix2 - ix1) * (iy2 - iy1)
    p_area = max((px2 - px1) * (py2 - py1), 1.0)
    return inter / p_area

class IDRemapper:
    def __init__(self, dist_thresh=80, frame_thresh=15):
        self.dist_thresh  = dist_thresh
        self.frame_thresh = frame_thresh
        self._lost: dict  = {}
        self._remap: dict = {}
    def canonical(self, tid): return self._remap.get(tid, tid)
    def update(self, fi, persons, prev_tids):
        current_tids = {p.track_id for p in persons}
        for tid in prev_tids - current_tids:
            canonical = self._remap.get(tid, tid)
            if canonical not in self._lost: self._lost[canonical] = (None, None, fi)
        expired = [tid for tid,(_, _, f) in self._lost.items() if fi-f > self.frame_thresh]
        for tid in expired: del self._lost[tid]
        new_mappings = {}; known_canonicals = set(self._remap.values())
        for p in persons:
            if p.track_id in self._remap or p.track_id in known_canonicals: continue
            best_match, best_dist = None, self.dist_thresh
            for old_tid, (cx, cy, _) in self._lost.items():
                if cx is None: continue
                dist = ((p.cx-cx)**2 + (p.cy-cy)**2)**0.5
                if dist < best_dist: best_dist=dist; best_match=old_tid
            if best_match is not None:
                self._remap[p.track_id]=best_match; new_mappings[p.track_id]=best_match
                del self._lost[best_match]
        return new_mappings
    def record_positions(self, persons):
        for p in persons:
            canonical = self._remap.get(p.track_id, p.track_id)
            if canonical in self._lost:
                cx, cy, f = self._lost[canonical]
                self._lost[canonical] = (p.cx, p.cy, f)

# ─── CrawlingRule  v6: AR=1.6, min_frames=10 ─────────────────
class CrawlingRule:
    def __init__(self, cfg):
        self.cfg=cfg; self._ar_buf=defaultdict(lambda: deque(maxlen=60))
        self._low_frames=defaultdict(int); self._triggered=set()
        self.debug=False; self._max_low=defaultdict(int)  # Fix4 diag

    def update(self, fi, persons, cached_gate=None):
        mf=self.cfg.crawl_min_frames; th=self.cfg.crawl_aspect_ratio_thresh
        out=[]; active={p.track_id for p in persons}
        for p in persons:
            if not near_gate_x(p, cached_gate, margin_ratio=0.3):
                if self.debug and fi%30==0:
                    print(f'[CRAWL] f={fi} ID={p.track_id} SKIP near_gate_x=False')
                self._low_frames[p.track_id]=0; continue
            self._ar_buf[p.track_id].append(p.aspect_ratio)
            median_ar=float(np.median(list(self._ar_buf[p.track_id])))
            if median_ar < th:
                self._low_frames[p.track_id]+=1
                self._max_low[p.track_id]=max(self._max_low[p.track_id], self._low_frames[p.track_id])
                if self.debug and self._low_frames[p.track_id]%5==0:
                    print(f'[CRAWL] f={fi} ID={p.track_id} AR={median_ar:.2f}<{th} cnt={self._low_frames[p.track_id]}/{mf}')
            else:
                cnt=self._low_frames[p.track_id]
                if cnt>=mf and p.track_id not in self._triggered:
                    self._triggered.add(p.track_id)
                    if self.debug:
                        print(f'[CRAWL ✓] f={fi} ID={p.track_id} TRIGGERED AR={median_ar:.2f} cnt={cnt}')
                    out.append(EventCandidate('crawling',fi-cnt,fi,[p.track_id],
                        min(1.0,cnt/(mf*3)),f'AR={median_ar:.2f}<{th} {cnt}프레임'))
                self._low_frames[p.track_id]=0
        for tid in list(self._low_frames):
            if tid not in active:
                cnt=self._low_frames[tid]
                if cnt>=mf and tid not in self._triggered:
                    self._triggered.add(tid)
                    out.append(EventCandidate('crawling',fi-cnt,fi,[tid],
                        min(1.0,cnt/(mf*3)),f'AR<{th} {cnt}프레임 후 시야이탈'))
                del self._low_frames[tid]
                if tid in self._ar_buf: del self._ar_buf[tid]
        return out

# ─── TailgatingRule v6 ───────────────────────────────────────
# [v5 대비 변경]
# 1) min_cooccupy_frames : 2 → 5 (FP 억제)
# 2) gate_overlap_thresh : 0.15 → 0.25 (더 엄격)
# 3) max_dist 필터 추가  : 두 사람 거리 > 200px 이면 pair 무시
#    → Normal 혼잡 영상에서 멀리 있는 사람 쌍(dist=400px 등) FP 제거
class TailgatingRule:
    def __init__(self, cfg):
        self.cfg=cfg
        self.min_cooccupy_frames=cfg.tailgating_min_frames   # 5
        self.gate_overlap_thresh=cfg.gate_overlap_thresh     # 0.25
        self.max_dist           =cfg.tailgate_max_dist       # 200px
        self._pair_streak: dict ={}; self._triggered: set=set()
        self._cy_history: dict  =defaultdict(lambda: deque(maxlen=30))
        self.debug=False

    def _movement_dir(self, tid) -> float:
        h=list(self._cy_history[tid])
        if len(h)<3: return 0.0
        return h[-1]-h[0]

    def _in_gate(self, p, gx1, gy1, gx2, gy2):
        ratio=_bbox_overlap_ratio(p.x1,p.y1,p.x2,p.y2,gx1,gy1,gx2,gy2)
        return ratio>=self.gate_overlap_thresh, ratio

    def update(self, fi, persons, fps, cached_gate=None):
        out=[]
        if cached_gate:
            gx1,gy1=cached_gate['x1'],cached_gate['y1']
            gx2,gy2=cached_gate['x2'],cached_gate['y2']
        else:
            gz=self.cfg.gate_zone; gx1,gy1,gx2,gy2=gz.x1,gz.y1,gz.x2,gz.y2

        in_gate_persons=[]; overlap_map={}
        for p in persons:
            inside,ratio=self._in_gate(p,gx1,gy1,gx2,gy2)
            overlap_map[p.track_id]=ratio
            if inside:
                in_gate_persons.append(p)
                self._cy_history[p.track_id].append(p.cy)

        if self.debug and fi%15==0:
            print(f'[TAIL DEBUG] frame={fi:5d}, persons={len(persons)}, '
                  f'in_gate={len(in_gate_persons)}, '
                  f'max_streak={max(self._pair_streak.values(),default=0)}')
            for p in persons:
                tag='IN' if p in in_gate_persons else '--'
                print(f'  [{tag}] id={p.track_id:3d} overlap={overlap_map[p.track_id]:.3f}')

        current_pairs=set()
        if len(in_gate_persons)>=2:
            for i in range(len(in_gate_persons)):
                for j in range(i+1,len(in_gate_persons)):
                    pa_c=in_gate_persons[i]; pb_c=in_gate_persons[j]
                    dist_ij=((pa_c.cx-pb_c.cx)**2+(pa_c.cy-pb_c.cy)**2)**0.5
                    if dist_ij>self.max_dist:     # ★ max_dist 필터
                        if self.debug:
                            print(f'[TAIL DEBUG] dist 필터: '
                                  f'ID{pa_c.track_id}-ID{pb_c.track_id} '
                                  f'{dist_ij:.0f}px>{self.max_dist}px → skip')
                        continue
                    current_pairs.add(frozenset({pa_c.track_id,pb_c.track_id}))

        for key in list(self._pair_streak.keys()):
            if key not in current_pairs: del self._pair_streak[key]

        for key in current_pairs:
            self._pair_streak[key]=self._pair_streak.get(key,0)+1
            streak=self._pair_streak[key]
            if streak>=self.min_cooccupy_frames and key not in self._triggered:
                ta,tb=list(key)
                pa=next((p for p in in_gate_persons if p.track_id==ta),in_gate_persons[0])
                pb=next((p for p in in_gate_persons if p.track_id==tb),in_gate_persons[1])
                dir_a=self._movement_dir(ta); dir_b=self._movement_dir(tb)
                if dir_a!=0.0 and dir_b!=0.0 and dir_a*dir_b<0:
                    if self.debug:
                        print(f'[TAIL DEBUG] 교행: ID{ta}(dir={dir_a:.1f}) '
                              f'vs ID{tb}(dir={dir_b:.1f}) → skip')
                    continue
                self._triggered.add(key)
                dist=((pa.cx-pb.cx)**2+(pa.cy-pb.cy)**2)**0.5
                dir_label='↓' if dir_a>=0 else '↑'
                ovlp_a=overlap_map.get(ta,0.0); ovlp_b=overlap_map.get(tb,0.0)
                out.append(EventCandidate(
                    'tailgating',fi-streak+1,fi,[ta,tb],0.75,
                    f'gate 동시 점유 {streak}f(min={self.min_cooccupy_frames}), '
                    f'방향({dir_label}), dist={dist:.0f}px, '
                    f'overlap=({ovlp_a:.2f}/{ovlp_b:.2f})'))
        return out

print('Rules 완료 (v6)')
print('  GateZoneConfig : tailgate_max_dist 필드 추가')
print('  CrawlingRule   : AR=1.6 / min_frames=10 (recall 회복)')
print('  TailgatingRule : min=5 / overlap>=0.25 / max_dist=200px 필터')

## 5. Gate 설정

CONFIG_DICT 좌표는 **gate 미탐지 시 fallback**으로만 사용됩니다.  
gate bbox가 탐지되면 `pass_line` / `crawl_zone` / `gate_zone`이 자동으로 gate 좌표에 맞게 보정됩니다.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 설정값 — 이 블록만 수정해서 파라미터를 바꿀 수 있습니다
# ═══════════════════════════════════════════════════════════════
CONFIG_DICT = {
    'camera_id': 'cam0', 'frame_width': 640, 'frame_height': 480,
    'gate_zone':  {'x1': 180, 'y1':  80, 'x2': 460, 'y2': 420},
    'pass_line':  {'x1': 180, 'y1': 280, 'x2': 460, 'y2': 280},
    'jump_zone':  {'x1': 160, 'y1':  40, 'x2': 480, 'y2': 180},
    'crawl_zone': {'x1': 160, 'y1': 340, 'x2': 480, 'y2': 430},

    # ── Jump ───────────────────────────────────────────────────
    'jump_min_frames':            3,

    # ── Crawling ───────────────────────────────────────────────
    'crawl_min_frames':          10,     # v5 15 → 10 (recall 회복)
    'crawl_aspect_ratio_thresh':  1.6,   # v5 1.55 → 1.6 (recall 회복)

    # ── Tailgating ─────────────────────────────────────────────
    'tailgate_time_window_s':    3.0,
    'tailgate_distance_thresh':  150.0,
    'tailgating_min_frames':       5,    # v5 2 → 5 (Normal FP 억제)
    'gate_overlap_thresh':         0.25, # v5 0.15 → 0.25 (더 엄격)
    'tailgate_max_dist':           200,  # NEW: 두 사람 거리(px) 초과 시 pair 무시
}

cfg = GateZoneConfig.from_dict(CONFIG_DICT)
print('Gate config 완료 (v6)')
print(f'  crawl_min_frames          = {cfg.crawl_min_frames}')
print(f'  crawl_aspect_ratio_thresh = {cfg.crawl_aspect_ratio_thresh}')
print(f'  tailgating_min_frames     = {cfg.tailgating_min_frames}')
print(f'  gate_overlap_thresh       = {cfg.gate_overlap_thresh}')
print(f'  tailgate_max_dist         = {cfg.tailgate_max_dist}')

In [ ]:
import matplotlib.pyplot as plt

PREVIEW_INDEX = 0
cap = cv2.VideoCapture(local_videos[PREVIEW_INDEX])
ret, frame = cap.read()
cap.release()

if ret:
    vis = frame.copy()
    pl = cfg.pass_line
    cv2.line(vis, (pl.x1, pl.y1), (pl.x2, pl.y2), (0, 255, 0), 2)
    cv2.putText(vis, 'PASS_LINE(fallback)', (pl.x1, pl.y1-5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0,255,0), 1)
    for z, c, lbl in [
        (cfg.crawl_zone, (220, 0, 0), 'CRAWL(fallback)'),
        (cfg.gate_zone,  (180, 180, 0), 'GATE_CFG(fallback)'),
    ]:
        cv2.rectangle(vis, (z.x1, z.y1), (z.x2, z.y2), c, 2)
        cv2.putText(vis, lbl, (z.x1, z.y1-4), cv2.FONT_HERSHEY_SIMPLEX, 0.4, c, 1)

    gate_det_preview = GateDetector(conf_thres=0.25)
    gate_bbox = gate_det_preview.detect_best(frame)
    if gate_bbox:
        gx1, gy1, gx2, gy2 = gate_bbox['x1'], gate_bbox['y1'], gate_bbox['x2'], gate_bbox['y2']
        gh = gy2 - gy1
        gate_cy = (gy1 + gy2) // 2
        # gate bbox
        cv2.rectangle(vis, (gx1, gy1), (gx2, gy2), (0, 220, 220), 3)
        cv2.putText(vis, f"GATE_DETECT {gate_bbox['conf']:.2f}", (gx1, gy1-18),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 220, 220), 2)
        # 자동 보정된 pass_line
        cv2.line(vis, (gx1, gate_cy), (gx2, gate_cy), (0, 255, 180), 2)
        cv2.putText(vis, 'PASS_LINE(auto)', (gx1, gate_cy-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0,255,180), 1)
        # 자동 보정된 crawl_zone
        cz_y1 = int(gy1 + gh * 0.6)
        cv2.rectangle(vis, (gx1, cz_y1), (gx2, gy2), (255, 100, 0), 2)
        cv2.putText(vis, 'CRAWL(auto)', (gx1, cz_y1+12), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255,100,0), 1)
        print(f'Gate 탐지: {gate_bbox}')
        print(f'  → pass_line y={gate_cy}, crawl_zone y1={cz_y1}')
    else:
        print('Gate 탐지 안 됨 → CONFIG_DICT fallback 사용')

    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.title(f'Zone 미리보기 — {os.path.basename(local_videos[PREVIEW_INDEX])}')
    plt.axis('off'); plt.show()

### 5-B. Gate 검증 시각화 — 영상 샘플별 gate bbox 탐지 확인

In [ ]:
import matplotlib.pyplot as plt

# 각 클래스에서 1개씩 샘플 추출 (최대 VERIFY_N개)
VERIFY_N = 8
VERIFY_FRAME = 30   # 각 영상의 몇 번째 프레임을 확인할지

verify_videos = []
seen_classes = set()
for vp in local_videos:
    fname = os.path.basename(vp).lower()
    cls = next((c for c in ['crawling', 'jump', 'tailgating', 'normal', 'nomal'] if c in fname), 'other')
    key = (cls, os.path.basename(os.path.dirname(vp)))
    if key not in seen_classes:
        seen_classes.add(key)
        verify_videos.append((vp, cls))
    if len(verify_videos) >= VERIFY_N:
        break

gate_det_v = GateDetector(conf_thres=0.25)
cols = min(4, len(verify_videos))
rows = (len(verify_videos) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3.5))
axes = np.array(axes).flatten() if len(verify_videos) > 1 else [axes]

detected_count = 0
for ax, (vp, cls) in zip(axes, verify_videos):
    cap = cv2.VideoCapture(vp)
    cap.set(cv2.CAP_PROP_POS_FRAMES, VERIFY_FRAME)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        ax.axis('off'); continue

    vis = frame.copy()
    gate = gate_det_v.detect_best(frame)
    if gate:
        detected_count += 1
        gx1, gy1, gx2, gy2 = gate['x1'], gate['y1'], gate['x2'], gate['y2']
        gh = gy2 - gy1
        cv2.rectangle(vis, (gx1, gy1), (gx2, gy2), (0, 220, 220), 3)
        cv2.putText(vis, f"conf={gate['conf']:.2f}", (gx1, gy1-8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 220, 220), 2)
        # pass_line
        gate_cy = (gy1 + gy2) // 2
        cv2.line(vis, (gx1, gate_cy), (gx2, gate_cy), (0, 255, 150), 2)
        # crawl_zone
        cz_y1 = int(gy1 + gh * 0.6)
        cv2.rectangle(vis, (gx1, cz_y1), (gx2, gy2), (255, 100, 0), 2)
        title_color = 'green'
    else:
        cv2.putText(vis, 'NO GATE', (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
        title_color = 'red'

    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(f'[{cls}] {os.path.basename(vp)[:25]}', fontsize=8, color=title_color)
    ax.axis('off')

for ax in axes[len(verify_videos):]:
    ax.axis('off')

plt.suptitle(f'Gate 탐지 검증: {detected_count}/{len(verify_videos)}개 성공', fontsize=12)
plt.tight_layout()
plt.show()
print(f'탐지율: {detected_count}/{len(verify_videos)} ({100*detected_count//len(verify_videos) if verify_videos else 0}%)')

## 6. 파이프라인 함수

**gate bbox 탐지 시 자동 좌표 보정**
- `pass_line` → gate 수평 중심선 (gate_cy)
- `crawl_zone` → gate 하단 40% 영역
- `gate_zone` → gate bbox 전체 (tailgating 판단 영역)

**jump 감지 구조**
1. 매 프레임 `JumpDetector.update()` → Y축 변위 누적
2. `JumpDetector.is_jump()` → True이면 jump 후보
3. gate 탐지된 경우: person cx가 gate x-range ±20% 안인지 확인
4. gate 없으면 필터 없이 통과

In [ ]:
import supervision as sv
from pathlib import Path
from tqdm.notebook import tqdm as tqdm_nb
import copy

def _save_clip(c, buf, out_dir, fps, margin_s=2.0):
    m = int(fps * margin_s)
    frames = [f for fi, f in buf if c.start_frame-m <= fi <= c.end_frame+m]
    if not frames: return None
    name = f"{c.event_type}_{c.start_frame}_{'_'.join(str(t) for t in c.track_ids)}.mp4"
    path = str(Path(out_dir) / name)
    h, w = frames[0].shape[:2]
    writer = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    for f in frames: writer.write(f)
    writer.release()
    return path

def _update_cfg_from_gate(cfg, gate):
    gx1, gy1, gx2, gy2 = gate['x1'], gate['y1'], gate['x2'], gate['y2']
    gh = gy2 - gy1
    cfg.pass_line  = PassLine(gx1, (gy1+gy2)//2, gx2, (gy1+gy2)//2)
    cfg.crawl_zone = Zone(gx1, int(gy1 + gh * 0.6), gx2, gy2)
    cfg.gate_zone  = Zone(gx1, gy1, gx2, gy2)

def _draw_overlay(frame, cfg, persons, gate_bbox=None, jump_scores=None, remapper=None):
    z = cfg.crawl_zone
    cv2.rectangle(frame, (z.x1, z.y1), (z.x2, z.y2), (180, 0, 0), 1)
    cv2.putText(frame, 'CRAWL', (z.x1+2, z.y1+12),
                cv2.FONT_HERSHEY_SIMPLEX, 0.38, (180,0,0), 1)
    if gate_bbox:
        gx1, gy1, gx2, gy2 = gate_bbox['x1'], gate_bbox['y1'], gate_bbox['x2'], gate_bbox['y2']
        cv2.rectangle(frame, (gx1, gy1), (gx2, gy2), (0, 220, 220), 2)
        cv2.putText(frame, f"gate {gate_bbox['conf']:.2f}", (gx1, gy1-6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 220, 220), 1, cv2.LINE_AA)
    else:
        cv2.putText(frame, 'gate: not found', (8, 48),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (80, 80, 200), 1)
    for p in persons:
        score = (jump_scores or {}).get(p.track_id, 0.0)
        is_jumping = score >= 0.25
        color = (0, 0, 255) if is_jumping else (0, 200, 0)
        cv2.rectangle(frame, (int(p.x1), int(p.y1)), (int(p.x2), int(p.y2)), color, 2 if is_jumping else 1)
        canonical = remapper.canonical(p.track_id) if remapper else p.track_id
        id_label = f'{canonical}' if canonical != p.track_id else f'{p.track_id}'
        label = f'ID:{id_label} J:{score:.2f}' if score > 0.05 else f'ID:{id_label}'
        cv2.putText(frame, label, (int(p.x1), int(p.y1)-4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.38, color, 1)

def run_pipeline(video_path, cfg_base, output_dir=OUTPUT_DIR, checkpoint=CHECKPOINT, conf=0.4,
                 gate_conf=0.25, near_gate_margin=0.2,
                 jump_multiplier=5.0, jump_min_score=0.25,
                 id_remap_frame_thresh=15,
                 verbose_rules=False, max_events=None):
    Path(output_dir+'/clips').mkdir(parents=True, exist_ok=True)
    cfg = copy.deepcopy(cfg_base)

    detector  = YOLODetector(conf_threshold=conf)
    gate_det  = GateDetector(conf_thres=gate_conf)           # Fix2
    tracker   = sv.ByteTrack()
    remapper  = IDRemapper(dist_thresh=80, frame_thresh=id_remap_frame_thresh)  # Fix2

    jump_detector  = JumpDetector(window_frames=30, jump_ratio=0.30,
                                  calibration_frames=90,
                                  dynamic_multiplier=jump_multiplier,  # Fix2
                                  min_score=jump_min_score)             # Fix2
    jump_triggered = set()
    jump_consec = defaultdict(int)   # 연속 jump 프레임 카운터 (FP 억제)

    rules = {
        'crawling':   CrawlingRule(cfg),
        'tailgating': TailgatingRule(cfg),
    }
    rules['tailgating'].debug = verbose_rules  # verbose_rules=True → [TAIL DEBUG] 로그 출력
    rules['crawling'].debug   = verbose_rules  # Fix4: [CRAWL] 탈락 사유 출력

    # ── Fix4: 진단 정보 수집 ──────────────────────────────────────
    diag_info = {
        'gate_found_frames': 0,
        'total_frames'     : 0,
        'unique_tids'      : set(),
        'id_switches'      : 0,
        'jump_peak_scores' : {},
        'tail_max_streak'  : 0,
    }
    cap   = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps   = cap.get(cv2.CAP_PROP_FPS) or 30.0
    buf   = deque(maxlen=int(fps * 20))
    all_c = []; ann_frames = []; fi = 0
    cached_gate = None; gate_cache_age = 0
    gate_calibrated = False
    prev_tids: set = set()

    for _ in tqdm_nb(range(total or 9999), desc=os.path.basename(video_path), leave=False):
        ret, frame = cap.read()
        if not ret: break
        fi += 1; buf.append((fi, frame.copy()))

        if max_events is not None and len(all_c) >= max_events:
            print(f'  [조기 종료] {max_events}건 달성 (frame={fi})')
            break

        clean = blur_faces(frame.copy())

        # person 탐지 + tracking
        raw = detector.detect_persons(clean)
        if raw:
            xyxy = np.array([[d.x1, d.y1, d.x2, d.y2] for d in raw], dtype=float)
            sv_d = sv.Detections(xyxy=xyxy, confidence=np.array([d.confidence for d in raw]))
        else:
            sv_d = sv.Detections.empty()
        sv_d = tracker.update_with_detections(sv_d)
        persons = [TrackedPerson(int(sv_d.tracker_id[i]), *[float(v) for v in sv_d.xyxy[i]])
                   for i in range(len(sv_d))] if sv_d.tracker_id is not None else []

        # ID switch 보정
        remapper.record_positions(persons)
        new_maps = remapper.update(fi, persons, prev_tids)
        for new_tid, old_tid in new_maps.items():
            jump_detector.transfer_history(old_tid, new_tid)
            if old_tid in jump_triggered:
                jump_triggered.add(new_tid)
        prev_tids = {p.track_id for p in persons}

        # gate bbox 탐지 + 캐시 + 좌표 자동 보정
        if gate_det.enabled:
            new_gate = gate_det.detect_best(clean)
            if new_gate:
                cached_gate = new_gate; gate_cache_age = 0
                diag_info['gate_found_frames'] += 1   # Fix4
                _update_cfg_from_gate(cfg, cached_gate)
                gate_h = cached_gate['y2'] - cached_gate['y1']
                jump_detector.set_gate_height(gate_h)
                if not gate_calibrated:
                    gate_calibrated = True
                    print(f'[{os.path.basename(video_path)}] gate 보정 → '
                          f'y={cfg.pass_line.y1}, jump_min={jump_detector._gate_min:.1f}px')
            else:
                gate_cache_age += 1
                if gate_cache_age > GATE_CACHE_FRAMES:
                    cached_gate = None

        # JumpDetector 업데이트
        active_tids = set()
        if sv_d.tracker_id is not None:
            for i, tid in enumerate(sv_d.tracker_id):
                tid = int(tid)
                active_tids.add(tid)
                diag_info['unique_tids'].add(remapper.canonical(tid))  # Fix4
                jump_detector.update(tid, sv_d.xyxy[i])
        jump_detector.cleanup(active_tids)
        jump_scores = {p.track_id: jump_detector.get_score(p.track_id) for p in persons}
        # Fix4: jump peak + tail streak 추적
        for p in persons:
            canon = remapper.canonical(p.track_id)
            sc = jump_scores.get(p.track_id, 0.0)
            if sc > diag_info['jump_peak_scores'].get(canon, 0.0):
                diag_info['jump_peak_scores'][canon] = sc
        if rules['tailgating']._pair_streak:
            diag_info['tail_max_streak'] = max(
                diag_info['tail_max_streak'],
                max(rules['tailgating']._pair_streak.values()))

        if verbose_rules and fi % 30 == 0 and persons:
            for p in persons:
                canon  = remapper.canonical(p.track_id)
                jsc    = jump_scores.get(p.track_id, 0.0)
                near_g = near_gate_x(p, cached_gate, 0.3)
                consec = jump_consec.get(canon, 0)
                in_trig= canon in jump_triggered
                print(f'  [VB] f={fi} ID={canon} ar={p.aspect_ratio:.2f} '
                      f'near_gate={near_g} jump_sc={jsc:.2f} '
                      f'consec={consec}/{cfg_base.jump_min_frames} triggered={in_trig}')

        cands = []

        # ── jump ────────────────────────────────────────────────
        # 3프레임 연속 is_jump()=True일 때만 트리거 (1프레임 FP 억제)
        for p in persons:
            canonical = remapper.canonical(p.track_id)
            if canonical in jump_triggered: continue
            if jump_detector.is_jump(p.track_id) and near_gate_x(p, cached_gate, margin_ratio=near_gate_margin):
                jump_consec[canonical] += 1
            else:
                jump_consec[canonical] = 0   # 조건 안 맞으면 리셋
            if jump_consec[canonical] < cfg_base.jump_min_frames: continue
            jump_triggered.add(canonical)
            score = jump_scores.get(p.track_id, 0.0)
            gate_info = f'gate_filter=ON conf={cached_gate["conf"]:.2f}' if cached_gate else 'gate_filter=OFF'
            cands.append(EventCandidate('jump', fi, fi, [canonical],
                                       min(1.0, score/0.5),
                                       f'Y(y2) score={score:.2f} consec=3 ({gate_info})'))

        # ── crawling / tailgating ────────────────────────────────
        cands += rules['crawling'].update(fi, persons, cached_gate)
        cands += rules['tailgating'].update(fi, persons, fps, cached_gate)

        for c in cands:
            if max_events is not None and len(all_c) >= max_events:
                break
            clip_path = _save_clip(c, buf, output_dir+'/clips', fps)

            # ── EfficientNet 2차 검증 ──────────────────────────────────
            eff = eff_verify(clip_path)
            c.efficientnet_result = eff
            if eff and eff["prediction"] == "normal":
                print(f'  [EFF✗] {c.event_type} → normal({eff["confidence"]:.2f}) 제거 ' +
                      str(os.path.basename(clip_path or "")))
                continue   # FP로 간주하고 기록 제외
            eff_tag = f' eff={eff["prediction"]}({eff["confidence"]:.2f})' if eff else ""
            # ──────────────────────────────────────────────────────────

            all_c.append(c)
            print(f'  [{c.event_type}] frame={fi} tracks={c.track_ids} '
                  f'conf={c.confidence:.2f}{eff_tag} | {c.reason}')
            vis = frame.copy()
            _draw_overlay(vis, cfg, persons, gate_bbox=cached_gate,
                         jump_scores=jump_scores, remapper=remapper)
            cv2.putText(vis, f'!! {c.event_type.upper()}', (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
            ann_frames.append((fi, c.event_type, vis.copy()))

    # ── Fix4: diag finalize ─────────────────────────────────────
    diag_info['total_frames'] = fi
    diag_info['id_switches']  = len(remapper._remap)
    if verbose_rules:
        gf = diag_info['gate_found_frames']
        print(f'  [DIAG] gate={gf}/{fi}f | '
              f'tracks={len(diag_info["unique_tids"])} | '
              f'id_sw={diag_info["id_switches"]}')
        pk = max(diag_info['jump_peak_scores'].values()) if diag_info['jump_peak_scores'] else 0.0
        print(f'  [DIAG] jump_peak={pk:.2f} | tail_max_streak={diag_info["tail_max_streak"]}')
    cap.release()
    with open(str(Path(output_dir)/'events.json'), 'w', encoding='utf-8') as f:
        json.dump([{'event_type': c.event_type, 'start_frame': c.start_frame,
                    'end_frame': c.end_frame, 'track_ids': c.track_ids,
                    'confidence': round(c.confidence, 4), 'reason': c.reason,
                    'status': c.status} for c in all_c],
                  f, indent=2, ensure_ascii=False)
    return all_c, ann_frames, diag_info

print('Pipeline 함수 완료 (v6)')
print('  IDRemapper 연결: ID switch 보정 + jump 히스토리 이어받기')
print('  set_gate_height: gate 탐지 시 jump threshold 하한 자동 갱신')
print('  활성 이벤트: jump / crawling / tailgating')
print('  verbose_rules=True 시 [TAIL DEBUG] 로그 출력')
print('  tailgating: min=5 / overlap>=0.25 / max_dist=200px')

## 7. 전체 영상 파이프라인 실행

In [ ]:
# ─── Fix1: 정답 라벨 = 상위 폴더명 기준 (cell-run보다 먼저 정의) ────
def get_gt_from_path(vp):
    """영상 경로에서 정답 라벨 추출. 상위 폴더명이 우선, fallback=파일명."""
    folder = os.path.basename(os.path.dirname(vp)).lower()
    if folder in {"jump", "crawling", "tailgating"}:
        return folder
    if folder in {"normal", "nomal"}:
        return "normal"
    fname = os.path.basename(vp).lower()
    for cls in ("jump", "crawling", "tailgating"):
        if cls in fname:
            return cls
    return "normal"

import json, datetime

def save_run_summary(label, candidates, vmap, target_videos,
                     settings=None, diag=None, out_dir=OUTPUT_DIR):
    """
    실행 결과 + 설정값을 pipeline_output/run_log.json 에 누적 저장.
    매번 덮어쓰지 않고 append → 여러 번 돌린 기록이 모두 남음.
    """
    EVENT_CLASSES = {'crawling', 'jump', 'tailgating'}

    tp = tn = fp = fn = 0
    per_video = []
    for vp in target_videos:
        gt  = get_gt_from_path(vp)   # Fix1: 폴더명 기준
        det = vmap.get(os.path.basename(vp), set())
        if gt == 'normal':
            status = 'TN' if not det else 'FP'
            if not det: tn += 1
            else:       fp += 1
        else:
            if gt in det: status = 'TP'; tp += 1
            else:         status = 'FN'; fn += 1
        per_video.append({'file': os.path.basename(vp), 'gt': gt,
                          'detected': sorted(det), 'status': status})

    total = tp + tn + fp + fn
    summary = {
        'label'    : label,
        'timestamp': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'settings' : settings or {
            'conf_threshold'       : 0.4,
            'gate_conf'            : 0.25,
            'near_gate_margin'     : 0.2,
            'jump_multiplier'      : 5.0,
            'jump_min_score'       : 0.25,
            'crawl_ar_thresh'      : CONFIG_DICT.get('crawl_aspect_ratio_thresh', 1.6),
            'crawl_min_frames'     : CONFIG_DICT.get('crawl_min_frames', 8),
            'tailgate_min_frames'  : 5,
            'id_remap_frame_thresh': 15,
            'gate_cache_frames'    : GATE_CACHE_FRAMES,
            'max_files_per_class'  : MAX_FILES_PER_CLASS,
        },
        'metrics': {
            'total': total, 'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn,
            'accuracy'  : round((tp+tn)/total*100, 1) if total else 0,
            'recall'    : round(tp/(tp+fn)*100, 1)   if (tp+fn) else 0,
            'precision' : round(tp/(tp+fp)*100, 1)   if (tp+fp) else 0,
        },
        'events_detected': len(candidates),
        'per_video': per_video,
        'diag_fn'  : diag or [],
    }

    log_path = os.path.join(out_dir, 'run_log.json')
    existing = []
    if os.path.exists(log_path):
        with open(log_path, 'r', encoding='utf-8') as f:
            try: existing = json.load(f)
            except: existing = []
    existing.append(summary)
    with open(log_path, 'w', encoding='utf-8') as f:
        json.dump(existing, f, indent=2, ensure_ascii=False)

    print(f'[run_log] {label} 저장 완료 → {log_path}')
    print(f'  Acc={summary["metrics"]["accuracy"]}%  '
          f'Recall={summary["metrics"]["recall"]}%  '
          f'Precision={summary["metrics"]["precision"]}%  '
          f'(TP={tp} TN={tn} FP={fp} FN={fn})')
    return summary

print('save_run_summary() 정의 완료')

In [ ]:
all_candidates = []
all_ann_frames = []
video_event_map = {}
video_diag_map  = {}   # Fix4: 영상별 진단 정보

MAX_FILES = None
target_videos = local_videos[:MAX_FILES] if MAX_FILES else local_videos
print(f'처리 대상: {len(target_videos)}/{len(local_videos)}개 파일')

for vp in tqdm_nb(target_videos, desc='영상 처리'):
    cands, frames, diag = run_pipeline(vp, cfg)   # Fix2: 3-tuple
    all_candidates.extend(cands)
    all_ann_frames.extend(frames)
    video_event_map[os.path.basename(vp)] = {c.event_type for c in cands}
    video_diag_map[os.path.basename(vp)]  = diag   # Fix4
    gt_label = get_gt_from_path(vp)                # Fix1
    print(f'GT={gt_label:12s} {os.path.basename(vp)}: {len(cands)}건 감지')

print(f'\n전체 감지: {len(all_candidates)}건')

# 결과 + 설정값 자동 저장
save_run_summary('baseline', all_candidates, video_event_map, target_videos)


## 8. 디버그 — 단일 영상
### 8-A. Jump Score 확인

In [ ]:
DEBUG_VIDEO_IDX  = 0
DEBUG_MAX_FRAMES = 500

video_path   = local_videos[DEBUG_VIDEO_IDX]
detector_d   = YOLODetector(conf_threshold=0.4)
gate_det_d   = GateDetector(conf_thres=0.25)
tracker_d    = sv.ByteTrack()
jump_det_d   = JumpDetector(window_frames=30, jump_ratio=0.30,
                             calibration_frames=90, dynamic_multiplier=3.0)

cached_gate_d = None; gate_cache_age_d = 0
cap = cv2.VideoCapture(video_path)
fps_d = cap.get(cv2.CAP_PROP_FPS) or 30.0

score_log  = []   # (frame, tid, score, is_jump)
debug_frames_d = []

for fi_d in range(DEBUG_MAX_FRAMES):
    ret, frame = cap.read()
    if not ret: break
    clean = blur_faces(frame.copy())

    raw = detector_d.detect_persons(clean)
    if raw:
        xyxy  = np.array([[d.x1, d.y1, d.x2, d.y2] for d in raw], dtype=float)
        sv_d2 = sv.Detections(xyxy=xyxy, confidence=np.array([d.confidence for d in raw]))
    else:
        sv_d2 = sv.Detections.empty()
    sv_d2 = tracker_d.update_with_detections(sv_d2)
    persons_d = [TrackedPerson(int(sv_d2.tracker_id[i]), *[float(v) for v in sv_d2.xyxy[i]])
                 for i in range(len(sv_d2))] if sv_d2.tracker_id is not None else []

    if gate_det_d.enabled:
        ng = gate_det_d.detect_best(clean)
        if ng: cached_gate_d = ng; gate_cache_age_d = 0
        else:
            gate_cache_age_d += 1
            if gate_cache_age_d > GATE_CACHE_FRAMES: cached_gate_d = None

    active = set()
    if sv_d2.tracker_id is not None:
        for i, tid in enumerate(sv_d2.tracker_id):
            active.add(int(tid))
            jump_det_d.update(int(tid), sv_d2.xyxy[i])
    jump_det_d.cleanup(active)

    cfg_d = copy.deepcopy(cfg)
    if cached_gate_d: _update_cfg_from_gate(cfg_d, cached_gate_d)

    for p in persons_d:
        score = jump_det_d.get_score(p.track_id)
        is_j  = jump_det_d.is_jump(p.track_id)
        score_log.append((fi_d+1, p.track_id, round(score, 3), is_j))

    if fi_d % 15 == 0:
        vis = frame.copy()
        jump_scores_d = {p.track_id: jump_det_d.get_score(p.track_id) for p in persons_d}
        _draw_overlay(vis, cfg_d, persons_d, gate_bbox=cached_gate_d, jump_scores=jump_scores_d)
        debug_frames_d.append(vis.copy())

cap.release()

score_log.sort(key=lambda x: -x[2])
print(f'상위 jump score (총 {len(score_log)}건 중 상위 20):')
for row in score_log[:20]:
    flag = '>>> JUMP' if row[3] else ''
    print(f'  frame={row[0]:4d} ID={row[1]} score={row[2]:.3f} {flag}')

In [ ]:
import matplotlib.pyplot as plt

show = debug_frames_d[:8]
cols = min(4, len(show)); rows = (len(show)+cols-1)//cols
if show:
    fig, axes = plt.subplots(rows, cols, figsize=(cols*4, rows*3))
    axes = np.array(axes).flatten() if len(show) > 1 else [axes]
    for ax, vis in zip(axes, show):
        ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); ax.axis('off')
    for ax in axes[len(show):]: ax.axis('off')
    plt.suptitle(f'Jump Score 디버그 — {os.path.basename(video_path)}', fontsize=11)
    plt.tight_layout(); plt.show()

### 8-B. Rule 상태 디버그 (tailgating/crawling/unpaid가 안 잡힐 때)

In [ ]:
# verbose_rules=True로 실행하면 30프레임마다 각 사람의 rule 조건 상태를 출력합니다
# 어떤 조건이 안 맞아서 이벤트가 안 잡히는지 확인용
DEBUG_RULE_VIDEO_IDX = 0

print(f'Rule 디버그: {os.path.basename(local_videos[DEBUG_RULE_VIDEO_IDX])}')
print('cy(line=N): 사람 중심 y / pass_line y')
print('ar: aspect_ratio (standing≈2~4, crawling<1.5)')
print('in_crawl: crawl_zone 안에 있는지')
print('in_gate: gate_zone 안에 있는지')
print('-'*60)

cands_d, _ = run_pipeline(
    local_videos[DEBUG_RULE_VIDEO_IDX],
    cfg,
    output_dir=OUTPUT_DIR,
    verbose_rules=True
)
print(f'\n감지 결과: {len(cands_d)}건')
for c in cands_d:
    print(f'  [{c.event_type}] {c.reason}')

### 8-C. yolo11n vs yolo11s 비교 (ID switch 빈도 + 속도)

In [ ]:
import time, csv

COMPARE_MAX_FRAMES = 300  # 영상당 비교 프레임 수

# normal / tailgating 영상 자동 선택 (각 클래스 최대 3개)
TARGET_CLASSES = {'normal', 'nomal', 'tailgating'}
compare_videos = []
for vp in local_videos:
    folder = os.path.basename(os.path.dirname(vp)).lower()
    # 로컬 복사 파일명에서 클래스 추출
    fname = os.path.basename(vp).lower()
    matched = any(cls in fname for cls in TARGET_CLASSES)
    if matched and len([v for v in compare_videos if any(cls in os.path.basename(v).lower() for cls in TARGET_CLASSES)]) < 6:
        compare_videos.append(vp)

if not compare_videos:
    # fallback: 앞에서 3개
    compare_videos = local_videos[:3]

print(f'비교 대상 영상 {len(compare_videos)}개:')
for v in compare_videos:
    print(f'  {os.path.basename(v)}')
print('-' * 60)

def run_tracker_compare(model_name, video_path, max_frames, conf=0.4):
    model   = YOLO(model_name)
    tracker = sv.ByteTrack(lost_track_buffer=30)
    cap = cv2.VideoCapture(video_path)
    all_tids, prev_tids = set(), set()
    id_switches = 0
    times = []

    for fi in range(max_frames):
        ret, frame = cap.read()
        if not ret:
            break
        t0 = time.perf_counter()
        results = model(frame, classes=[0], conf=conf, verbose=False)
        dets = []
        for r in results:
            if r.boxes is None:
                continue
            for box in r.boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                dets.append([x1, y1, x2, y2, float(box.conf[0])])
        if dets:
            xyxy  = np.array([[d[0], d[1], d[2], d[3]] for d in dets])
            confs = np.array([d[4] for d in dets])
            sv_d  = sv.Detections(xyxy=xyxy, confidence=confs)
        else:
            sv_d = sv.Detections.empty()
        sv_d = tracker.update_with_detections(sv_d)
        times.append(time.perf_counter() - t0)

        curr_tids = set(int(t) for t in sv_d.tracker_id) if sv_d.tracker_id is not None else set()
        if prev_tids:
            id_switches += len(curr_tids - all_tids)
        all_tids.update(curr_tids)
        prev_tids = curr_tids

    cap.release()
    avg_ms = float(np.mean(times)) * 1000 if times else 0
    return {
        'video'           : os.path.basename(video_path),
        'model'           : model_name,
        'total_unique_ids': len(all_tids),
        'id_switches'     : id_switches,
        'avg_ms'          : round(avg_ms, 1),
        'fps'             : round(1000 / avg_ms, 1) if avg_ms > 0 else 0,
        'frames'          : len(times),
    }

all_results = []
models = ['yolo11n.pt', 'yolo11s.pt']

for vp in compare_videos:
    print(f'\n[{os.path.basename(vp)}]')
    for model_name in models:
        print(f'  {model_name} ...', end=' ', flush=True)
        r = run_tracker_compare(model_name, vp, COMPARE_MAX_FRAMES)
        all_results.append(r)
        print(f'ID={r["total_unique_ids"]} switch={r["id_switches"]} {r["avg_ms"]}ms')

# 요약 출력
print('\n' + '=' * 65)
print(f'{"영상":<30} {"모델":<12} {"고유ID":>6} {"switch":>7} {"ms":>7} {"FPS":>6}')
print('-' * 65)
for r in all_results:
    print(f'{r["video"][:30]:<30} {r["model"]:<12} {r["total_unique_ids"]:>6} '
          f'{r["id_switches"]:>7} {r["avg_ms"]:>6.1f}ms {r["fps"]:>5.1f}')

# Drive에 CSV 저장
csv_path = f'{DRIVE_SAVE}/model_compare.csv'
os.makedirs(DRIVE_SAVE, exist_ok=True)
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=all_results[0].keys())
    writer.writeheader()
    writer.writerows(all_results)
print(f'\nCSV 저장 → {csv_path}')

# n vs s 평균 비교
for model_name in models:
    rows = [r for r in all_results if r['model'] == model_name]
    if rows:
        avg_id  = sum(r['total_unique_ids'] for r in rows) / len(rows)
        avg_sw  = sum(r['id_switches'] for r in rows) / len(rows)
        avg_fps = sum(r['fps'] for r in rows) / len(rows)
        print(f'{model_name}: 평균 고유ID={avg_id:.1f} switch={avg_sw:.1f} FPS={avg_fps:.1f}')


## 9. 결과 확인

In [ ]:
import pandas as pd
from collections import Counter

if all_candidates:
    df = pd.DataFrame([{
        'event_type': c.event_type, 'start': c.start_frame, 'end': c.end_frame,
        'tracks': str(c.track_ids), 'conf': round(c.confidence, 3),
        'status': c.status, 'reason': c.reason
    } for c in all_candidates])
    display(df)
    cnt = Counter(c.event_type for c in all_candidates)
    plt.figure(figsize=(8, 3))
    plt.bar(cnt.keys(), cnt.values(), color=['#e74c3c', '#e67e22', '#3498db', '#2ecc71'])
    plt.title('이벤트 유형별'); plt.ylabel('count'); plt.tight_layout(); plt.show()
else:
    print('감지된 이벤트 없음')

In [ ]:
if all_ann_frames:
    show = all_ann_frames[:8]
    cols = min(4, len(show)); rows = (len(show)+cols-1)//cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*4, rows*3))
    axes = np.array(axes).flatten() if len(show) > 1 else [axes]
    for ax, (fi, et, vis) in zip(axes, show):
        ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
        ax.set_title(f'{et.upper()} @{fi}', fontsize=9); ax.axis('off')
    for ax in axes[len(show):]: ax.axis('off')
    plt.tight_layout(); plt.show()

### 9-B. 폴더명(정답) vs 감지 결과 매칭 확인

In [ ]:
# ─── Fix 1: 정답 라벨 = 상위 폴더명 기준 ────────────────────────────
def get_gt_from_path(vp):
    """영상 경로에서 정답 라벨 추출. 상위 폴더명이 우선, fallback=파일명."""
    folder = os.path.basename(os.path.dirname(vp)).lower()
    if folder in {'jump', 'crawling', 'tailgating'}:
        return folder
    if folder in {'normal', 'nomal'}:          # 오타 'nomal' 포함
        return 'normal'
    # fallback: 파일명 기반
    fname = os.path.basename(vp).lower()
    for cls in ('jump', 'crawling', 'tailgating'):
        if cls in fname:
            return cls
    return 'normal'

EVENT_CLASSES = {'crawling', 'jump', 'tailgating'}

match = mismatch = fn = fp = 0
rows = []
fp_files = []; fn_files = []; wrong_files = []

for vp in target_videos:
    gt       = get_gt_from_path(vp)          # ★ 폴더명 기준
    detected = video_event_map.get(os.path.basename(vp), set())
    diag     = video_diag_map.get(os.path.basename(vp), {})

    if gt == 'normal':
        if not detected:
            status = '✅ TN'; match += 1
        else:
            status = f'❌ FP ({chr(44).join(sorted(detected))})'; fp += 1
            fp_files.append((os.path.basename(vp), detected, diag))
    else:
        if gt in detected:
            status = '✅ TP'; match += 1
        elif detected:
            status = f'⚠️  WRONG ({chr(44).join(sorted(detected))})'; mismatch += 1
            wrong_files.append((os.path.basename(vp), gt, detected, diag))
        else:
            status = '❌ FN (미탐지)'; fn += 1
            fn_files.append((os.path.basename(vp), gt, diag))

    rows.append((status, gt, ', '.join(sorted(detected)) if detected else '없음', os.path.basename(vp)))

for status, gt, det, fname in rows:
    print(f'{status:32s} GT={gt:12s} det={det:22s} {fname[:35]}')

total = len(rows)
print(f'\n총 {total}개 영상')
print(f'  ✅ 정답 (TP+TN): {match} ({100*match//total if total else 0}%)')
print(f'  ❌ FN 미탐지   : {fn}')
print(f'  ❌ FP 오탐     : {fp}')
print(f'  ⚠️  Wrong class : {mismatch}')

# ── Fix 4: 실패 파일 + 진단 정보 출력 ─────────────────────────────
if fn_files:
    print(f'\n─── FN 목록 ({len(fn_files)}건) ──────────────────────────────')
    for fname, gt, d in fn_files:
        gate_ok  = f'gate={d.get("gate_found_frames",0)}/{d.get("total_frames",0)}f'
        tracks   = f'tracks={len(d.get("unique_tids",set()))}'
        switches = f'id_sw={d.get("id_switches",0)}'
        pk_val   = max(d['jump_peak_scores'].values()) if d.get('jump_peak_scores') else 0.0
        jpeak    = f'jump_peak={pk_val:.2f}'
        tsteak   = f'tail_max={d.get("tail_max_streak",0)}'
        print(f'  GT={gt:12s} | {gate_ok} {tracks} {switches} {jpeak} {tsteak} | {fname}')

if fp_files:
    print(f'\n─── FP 목록 ({len(fp_files)}건) ──────────────────────────────')
    for fname, det, d in fp_files:
        gate_ok  = f'gate={d.get("gate_found_frames",0)}/{d.get("total_frames",0)}f'
        tracks   = f'tracks={len(d.get("unique_tids",set()))}'
        print(f'  det={chr(44).join(sorted(det)):20s} | {gate_ok} {tracks} | {fname}')

if wrong_files:
    print(f'\n─── WRONG 목록 ({len(wrong_files)}건) ─────────────────────────')
    for fname, gt, det, d in wrong_files:
        print(f'  GT={gt:12s} det={chr(44).join(sorted(det)):20s} | {fname}')


### 9-C. FP 분석 — Normal 오탐 원인 파악

`run_pipeline` 결과에서 Normal 영상에 감지된 이벤트를 타입별로 분해하고,
각 이벤트 `reason` 필드를 파싱해 오탐 원인과 임계값 조정 방향을 자동 제안합니다.

In [ ]:
import re
from collections import defaultdict

# ── Normal FP 영상 + EventCandidate 수집 ────────────────────────────
fp_normal_videos = set()
for vp in target_videos:
    if get_gt_from_path(vp) == 'normal':
        det = video_event_map.get(os.path.basename(vp), set())
        if det:
            fp_normal_videos.add(os.path.basename(vp))

# all_candidates 에서 FP 영상에 해당하는 이벤트만 필터
fp_candidates_by_evt = defaultdict(list)   # evt → [(video_name, EventCandidate, diag)]
for c in all_candidates:
    # reason 안에 파일명이 없으므로 clip 경로 또는 track_ids로 역추적이 어렵다
    # → video_event_map에서 FP 영상만 뽑아 target_videos 순서로 매칭
    pass

# 더 직접적인 방법: (vp, EventCandidate) 쌍으로 run_pipeline 재실행 없이 재구성
# cell-run에서 per-video cands를 분리 저장하지 않았으므로
# video_event_map + video_diag_map으로 통계만 집계

fp_by_evt = defaultdict(list)   # evt → [(basename, diag)]
for vp in target_videos:
    if get_gt_from_path(vp) != 'normal': continue
    detected = video_event_map.get(os.path.basename(vp), set())
    if not detected: continue
    diag = video_diag_map.get(os.path.basename(vp), {})
    for evt in detected:
        fp_by_evt[evt].append((os.path.basename(vp), diag))

total_fp_cnt = sum(len(v) for v in fp_by_evt.values())
fp_video_cnt = len([vp for vp in target_videos
                    if get_gt_from_path(vp) == 'normal'
                    and video_event_map.get(os.path.basename(vp))])
print(f'Normal FP: {fp_video_cnt}개 영상 / 이벤트 {total_fp_cnt}건')
print()

for evt in ['jump', 'crawling', 'tailgating']:
    items = fp_by_evt.get(evt, [])
    if not items:
        print(f'[{evt.upper():10s}] FP 없음')
        continue

    print(f'══ {evt.upper()} FP: {len(items)}건 ═══════════════════════════════════════')
    jpeaks = []
    tail_maxes = []
    gate_rates = []
    id_sws = []

    for fname, d in items:
        tf        = max(d.get('total_frames', 1), 1)
        gf        = d.get('gate_found_frames', 0)
        gate_rate = gf / tf
        n_tracks  = len(d.get('unique_tids', set()))
        id_sw     = d.get('id_switches', 0)
        pk        = max(d['jump_peak_scores'].values()) if d.get('jump_peak_scores') else 0.0
        t_streak  = d.get('tail_max_streak', 0)

        gate_rates.append(gate_rate)
        jpeaks.append(pk)
        tail_maxes.append(t_streak)
        id_sws.append(id_sw)

        print(f'  gate={gf}/{tf}f({gate_rate:.0%}) tracks={n_tracks} '
              f'id_sw={id_sw} jump_peak={pk:.2f} tail_streak={t_streak}  {fname}')

    print()
    # ── 원인 진단 ────────────────────────────────────────────────
    avg_gate = sum(gate_rates)/len(gate_rates)
    avg_ids  = sum(id_sws)/len(id_sws)

    if evt == 'jump':
        avg_pk = sum(jpeaks)/len(jpeaks)
        below35 = sum(1 for v in jpeaks if v < 0.35)
        below40 = sum(1 for v in jpeaks if v < 0.40)
        print(f'  [JUMP 분석] jump_peak: min={min(jpeaks):.2f} avg={avg_pk:.2f} max={max(jpeaks):.2f}')
        print(f'             peak < 0.35: {below35}/{len(jpeaks)}건  peak < 0.40: {below40}/{len(jpeaks)}건')
        if below35 >= len(jpeaks) * 0.6:
            print(f'  ★ 제안: JUMP_MIN_SCORE 0.25 → 0.35 (FP {below35}건 제거 예상)')
        elif below40 >= len(jpeaks) * 0.6:
            print(f'  ★ 제안: JUMP_MIN_SCORE 0.25 → 0.40 (FP {below40}건 제거 예상)')
        else:
            print(f'  ★ jump_peak가 높음 → 실제로 사람이 뛰는 모션 존재 → EfficientNet 필터 필요')

    elif evt == 'crawling':
        print(f'  [CRAWL 분석] gate_rate: avg={avg_gate:.0%}  id_switches: avg={avg_ids:.1f}')
        if avg_gate < 0.3:
            print(f'  ★ gate가 자주 안 잡힘 → gate 없을 때 near_gate_x 필터가 fallback zone 기준으로 동작')
            print(f'     → gate_conf 낮추거나 gate_best.pt 재학습 고려')
        else:
            print(f'  ★ gate는 정상 탐지 → crawl_min_frames 올리거나 AR thresh 높이기')
            print(f'     또는 EfficientNet이 normal로 판정해 제거되는지 확인')

    elif evt == 'tailgating':
        avg_streak = sum(tail_maxes)/len(tail_maxes)
        below8 = sum(1 for v in tail_maxes if v < 8)
        below10 = sum(1 for v in tail_maxes if v < 10)
        print(f'  [TAIL 분석] tail_streak: min={min(tail_maxes)} avg={avg_streak:.1f} max={max(tail_maxes)}')
        print(f'             streak < 8: {below8}/{len(tail_maxes)}건  streak < 10: {below10}/{len(tail_maxes)}건')
        if below8 >= len(tail_maxes) * 0.5:
            print(f'  ★ 제안: TAILGATE_MIN_FRAMES 5 → 8 (FP {below8}건 제거 예상)')
        elif below10 >= len(tail_maxes) * 0.5:
            print(f'  ★ 제안: TAILGATE_MIN_FRAMES 5 → 10 (FP {below10}건 제거 예상)')
        else:
            print(f'  ★ streak이 높음 → 실제로 두 사람이 오래 겹쳐 있음 → dist/방향 필터 강화 or EfficientNet')

    print()


#### FP 원인별 임계값 시뮬레이션

아래 셀에서 Normal 필터 강화 후 TP/TN/FP/FN 변화를 시뮬레이션합니다.
실제 재실행 없이 `video_diag_map`과 `video_event_map`만으로 계산합니다.

In [ ]:
# ── 필터 파라미터 (여기서만 수정) ──────────────────────────────────
SIM_JUMP_MIN_SCORE_STRICT    = 0.35   # 현재 0.25
SIM_TAILGATE_MIN_FRAMES_STRICT = 8    # 현재 5
SIM_CRAWL_MIN_FRAMES_STRICT  = 12    # 현재 10
# ───────────────────────────────────────────────────────────────────

def sim_normal_filter(vp, detected, diag,
                      jump_score_thresh=SIM_JUMP_MIN_SCORE_STRICT,
                      tail_frames_thresh=SIM_TAILGATE_MIN_FRAMES_STRICT,
                      crawl_frames_thresh=SIM_CRAWL_MIN_FRAMES_STRICT):
    """
    diag만으로 이 영상이 강화 필터를 통과할지 시뮬레이션.
    반환: 통과하는 이벤트 집합 (빈 set이면 TN 처리)
    """
    passed = set()
    for evt in detected:
        if evt == 'jump':
            pk = max(diag['jump_peak_scores'].values()) if diag.get('jump_peak_scores') else 0.0
            if pk >= jump_score_thresh:
                passed.add(evt)
        elif evt == 'tailgating':
            if diag.get('tail_max_streak', 0) >= tail_frames_thresh:
                passed.add(evt)
        elif evt == 'crawling':
            # crawl_max_consec 정보가 diag에 없으므로 EfficientNet 의존 표시
            passed.add('crawling_eff_dep')  # EfficientNet에 위임
    return passed

# ── 시뮬레이션 실행 ───────────────────────────────────────────────
tp_base = tp_sim = 0
tn_base = tn_sim = 0
fp_base = fp_sim = 0
fn_base = fn_sim = 0

removed_fp = []; survived_fp = []

for vp in target_videos:
    gt       = get_gt_from_path(vp)
    detected = video_event_map.get(os.path.basename(vp), set())
    diag     = video_diag_map.get(os.path.basename(vp), {})

    # ── Baseline ──
    if gt == 'normal':
        if not detected: tn_base += 1
        else:            fp_base += 1
    else:
        if gt in detected: tp_base += 1
        else:              fn_base += 1

    # ── Simulated ──
    if gt == 'normal':
        passed = sim_normal_filter(vp, detected, diag)
        passed_real = {e for e in passed if e != 'crawling_eff_dep'}
        if not passed_real:
            tn_sim += 1
            if detected: removed_fp.append((os.path.basename(vp), detected, diag))
        else:
            fp_sim += 1
            survived_fp.append((os.path.basename(vp), passed_real, diag))
    else:
        passed = sim_normal_filter(vp, detected, diag)
        if gt in detected:
            # TP였던 것도 필터가 제거할 수 있음 — 확인
            if gt == 'jump':
                pk = max(diag['jump_peak_scores'].values()) if diag.get('jump_peak_scores') else 0.0
                if pk >= SIM_JUMP_MIN_SCORE_STRICT: tp_sim += 1
                else: fn_sim += 1  # 필터가 TP도 제거
            elif gt == 'tailgating':
                if diag.get('tail_max_streak',0) >= SIM_TAILGATE_MIN_FRAMES_STRICT: tp_sim += 1
                else: fn_sim += 1
            else:
                tp_sim += 1  # crawling은 EfficientNet 위임 → TP 유지
        else:
            fn_sim += 1

total = len(target_videos)
print('=' * 55)
print(f'{"":20s} {"Baseline":>12} {"Filtered":>12}')
print('=' * 55)
for k, bv, sv in [('TP', tp_base, tp_sim), ('TN', tn_base, tn_sim),
                   ('FP', fp_base, fp_sim), ('FN', fn_base, fn_sim)]:
    mark = ' ✅' if (k in ('TP','TN') and sv >= bv) else \
           ' ✅' if (k in ('FP','FN') and sv <= bv) else \
           ' ❌' if (k in ('TP','TN') and sv < bv) else \
           ' ❌' if (k in ('FP','FN') and sv > bv) else ''
    print(f'{k:20s} {bv:>12} {sv:>12}{mark}')

acc_b  = round((tp_base+tn_base)/(tp_base+tn_base+fp_base+fn_base)*100,1)
acc_s  = round((tp_sim +tn_sim) /(tp_sim +tn_sim +fp_sim +fn_sim )*100,1)
prec_b = round(tp_base/(tp_base+fp_base)*100,1) if (tp_base+fp_base) else 0
prec_s = round(tp_sim /(tp_sim +fp_sim )*100,1) if (tp_sim +fp_sim ) else 0
rec_b  = round(tp_base/(tp_base+fn_base)*100,1) if (tp_base+fn_base) else 0
rec_s  = round(tp_sim /(tp_sim +fn_sim )*100,1) if (tp_sim +fn_sim ) else 0
print(f'{"Accuracy":20s} {acc_b:>11}% {acc_s:>11}%')
print(f'{"Precision":20s} {prec_b:>11}% {prec_s:>11}%')
print(f'{"Recall":20s} {rec_b:>11}% {rec_s:>11}%')
print()

if removed_fp:
    print(f'[제거된 FP] {len(removed_fp)}건:')
    for fname, det, d in removed_fp:
        pk = max(d['jump_peak_scores'].values()) if d.get('jump_peak_scores') else 0.0
        print(f'  det={sorted(det)} jump_peak={pk:.2f} tail_streak={d.get("tail_max_streak",0)}  {fname}')

if survived_fp:
    print(f'\n[남은 FP] {len(survived_fp)}건 → 추가 조치 필요:')
    for fname, det, d in survived_fp:
        print(f'  det={sorted(det)}  {fname}')
    print()
    print('[다음 단계 제안]')
    evt_counts = {}
    for _, det, _ in survived_fp:
        for e in det: evt_counts[e] = evt_counts.get(e, 0) + 1
    for evt, cnt in sorted(evt_counts.items(), key=lambda x: -x[1]):
        if evt == 'jump':
            print(f'  JUMP  {cnt}건 남음 → 3프레임 연속 조건 이미 있음 → EfficientNet 강화 (EFF_CONF_THRESHOLD 낮추기)')
        elif evt == 'crawling':
            print(f'  CRAWL {cnt}건 남음 → EfficientNet이 주요 필터 (crawling vs normal 판정 확인)')
        elif evt == 'tailgating':
            print(f'  TAIL  {cnt}건 남음 → 실제 영상 재생으로 "왜 두 사람이 겹쳤는지" 확인 후 데이터 수집')


#### 9-C 분석 후 다음 단계

| 상황 | 조치 |
|------|------|
| Jump FP peak < 0.35 다수 | `JUMP_MIN_SCORE` 올리기 → section 10-B 재실행 |
| Jump FP peak ≥ 0.35 다수 | EfficientNet `EFF_CONF_THRESHOLD` 낮추기 (0.50→0.40) |
| Tailgating FP streak < 8 | `TAILGATE_MIN_FRAMES` 올리기 → section 10-B 재실행 |
| Tailgating FP streak ≥ 8 | 데이터 수집 필요 (normal vs tailgating 경계 케이스) |
| Crawling FP 다수 | `CRAWL_MIN_FRAMES` / `CRAWL_AR_THRESH` 조정 + EfficientNet 확인 |
| Normal FP 줄었지만 Recall 하락 | EfficientNet 재학습 (더 많은 데이터로) 고려 |


## 10. FN 진단 — 놓친 영상 원인 분류

FN 영상만 다시 돌려서 왜 놓쳤는지 자동 분류합니다.

**원인 분류 기준 (순서대로 판정)**
1. `person_detection` — 영상 전체에서 사람이 한 번도 탐지 안 됨
2. `gate_detection` — gate bbox 한 번도 탐지 안 됨
3. `gate_area_filter` — 사람은 잡혔는데 gate x-range 조건 탈락
4. `rule_threshold` — gate 근처까지 왔는데 rule 조건 미달 (score/AR/streak 로그 포함)
5. `id_switch_suspected` — 고유 ID 수 과다 (실제 사람 수 대비 2배 이상)

In [ ]:
def run_pipeline_diag(video_path, cfg_base, gt_class, conf=0.4):
    """
    FN 영상을 재처리하며 실패 원인을 진단한다.
    반환: dict — cause, detail, 각종 max 지표
    """
    import copy
    cfg = copy.deepcopy(cfg_base)

    detector     = YOLODetector(conf_threshold=conf)
    gate_det     = GateDetector(conf_thres=0.25)
    tracker      = sv.ByteTrack()
    jump_det     = JumpDetector(window_frames=30, calibration_frames=90,
                                dynamic_multiplier=5.0, min_score=0.25)

    cap   = cv2.VideoCapture(video_path)
    fps   = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # 진단용 누적 변수
    person_counts      = []   # 프레임별 탐지 인원
    gate_det_frames    = 0    # gate 탐지된 프레임 수
    near_gate_per_tid  = defaultdict(int)   # tid → gate 근처 프레임 수
    jump_score_per_tid = defaultdict(float) # tid → 최대 jump score
    ar_per_tid         = defaultdict(list)  # tid → AR 리스트
    cooccupy_streak    = 0    # tailgating용 최대 연속 점유 프레임
    _streak_cur        = 0
    unique_ids         = set()
    cached_gate        = None; gate_cache_age = 0

    for fi in range(total or 9999):
        ret, frame = cap.read()
        if not ret: break

        clean = blur_faces(frame.copy())

        raw = detector.detect_persons(clean)
        person_counts.append(len(raw))

        if raw:
            xyxy = np.array([[d.x1, d.y1, d.x2, d.y2] for d in raw], dtype=float)
            sv_d = sv.Detections(xyxy=xyxy, confidence=np.array([d.confidence for d in raw]))
        else:
            sv_d = sv.Detections.empty()
        sv_d = tracker.update_with_detections(sv_d)
        persons = [TrackedPerson(int(sv_d.tracker_id[i]), *[float(v) for v in sv_d.xyxy[i]])
                   for i in range(len(sv_d))] if sv_d.tracker_id is not None else []

        for p in persons:
            unique_ids.add(p.track_id)

        # gate 탐지
        if gate_det.enabled:
            ng = gate_det.detect_best(clean)
            if ng:
                cached_gate = ng; gate_cache_age = 0
                _update_cfg_from_gate(cfg, cached_gate)
                jump_det.set_gate_height(ng['y2'] - ng['y1'])
                gate_det_frames += 1
            else:
                gate_cache_age += 1
                if gate_cache_age > GATE_CACHE_FRAMES:
                    cached_gate = None

        # jump score, AR, near_gate 누적
        if sv_d.tracker_id is not None:
            for i, tid in enumerate(sv_d.tracker_id):
                tid = int(tid)
                jump_det.update(tid, sv_d.xyxy[i])

        for p in persons:
            score = jump_det.get_score(p.track_id)
            jump_score_per_tid[p.track_id] = max(jump_score_per_tid[p.track_id], score)
            ar_per_tid[p.track_id].append(p.aspect_ratio)
            if near_gate_x(p, cached_gate, margin_ratio=0.2):
                near_gate_per_tid[p.track_id] += 1

        # tailgating streak
        if cached_gate:
            gx1, gy1, gx2, gy2 = cached_gate['x1'], cached_gate['y1'], cached_gate['x2'], cached_gate['y2']
            in_gate = [p for p in persons if gx1 <= p.cx <= gx2 and gy1 <= p.cy <= gy2]
            if len(in_gate) >= 2:
                _streak_cur += 1
                cooccupy_streak = max(cooccupy_streak, _streak_cur)
            else:
                _streak_cur = 0

        jump_det.cleanup({p.track_id for p in persons})

    cap.release()

    # ── 원인 판정 ──────────────────────────────────────────────
    max_persons   = max(person_counts) if person_counts else 0
    any_near_gate = any(v > 0 for v in near_gate_per_tid.values())
    max_jump      = max(jump_score_per_tid.values()) if jump_score_per_tid else 0.0
    min_ar        = min(np.median(v) for v in ar_per_tid.values()) if ar_per_tid else 99.0
    jump_thresh_est = jump_det._dynamic_threshold or jump_det._gate_min

    if max_persons == 0:
        cause = 'person_detection'
        detail = '전체 프레임에서 사람 미탐지'
    elif gate_det_frames == 0:
        cause = 'gate_detection'
        detail = f'gate bbox 미탐지 (전체 {total}f)'
    elif not any_near_gate:
        cause = 'gate_area_filter'
        detail = f'사람 탐지됨(max={max_persons}) but gate x-range 조건 탈락'
    elif gt_class == 'jump' and max_jump < 0.25:
        cause = 'rule_threshold'
        detail = f'jump max_score={max_jump:.3f} < min_score=0.25'
    elif gt_class == 'jump' and max_jump >= 0.25:
        cause = 'rule_threshold'
        detail = f'jump score OK({max_jump:.3f}) but threshold 미달 (est={jump_thresh_est:.1f}px)'
    elif gt_class == 'crawling' and min_ar > cfg.crawl_aspect_ratio_thresh:
        cause = 'rule_threshold'
        detail = f'crawling min_AR={min_ar:.2f} > thresh={cfg.crawl_aspect_ratio_thresh}'
    elif gt_class == 'tailgating' and cooccupy_streak < 5:
        cause = 'rule_threshold'
        detail = f'tailgating max_streak={cooccupy_streak} < 5'
    elif len(unique_ids) > max(3, total / fps * 2):
        cause = 'id_switch_suspected'
        detail = f'고유 ID={len(unique_ids)}개 (영상 {total/fps:.0f}s, 기대치 대비 과다)'
    else:
        cause = 'rule_threshold'
        detail = f'원인 불명 — jump={max_jump:.3f} AR={min_ar:.2f} streak={cooccupy_streak}'

    return {
        'video'         : os.path.basename(video_path),
        'gt'            : gt_class,
        'cause'         : cause,
        'detail'        : detail,
        'max_persons'   : max_persons,
        'gate_det_f'    : gate_det_frames,
        'max_jump_score': round(max_jump, 3),
        'min_AR'        : round(min_ar, 3),
        'max_streak'    : cooccupy_streak,
        'unique_ids'    : len(unique_ids),
        'total_frames'  : total,
    }

print('run_pipeline_diag() 정의 완료')

In [ ]:
import csv
from collections import Counter

EVENT_CLASSES = {'crawling', 'jump', 'tailgating'}

# 9-B에서 만든 rows 재사용: FN 영상만 추출
fn_videos = []
for vp in target_videos:
    gt = get_gt_from_path(vp)   # Fix1: 폴더명 기준
    if gt == 'normal':
        continue
    detected = video_event_map.get(os.path.basename(vp), set())
    if gt not in detected:
        fn_videos.append((vp, gt))

print(f'FN 영상: {len(fn_videos)}개')
for vp, gt in fn_videos:
    print(f'  [{gt}] {os.path.basename(vp)}')

# FN 영상 진단 실행
diag_results = []
for vp, gt in tqdm_nb(fn_videos, desc='FN 진단'):
    result = run_pipeline_diag(vp, cfg, gt)
    diag_results.append(result)
    print(f'  [{result["cause"]}] {result["video"][:40]} — {result["detail"]}')

# 원인별 집계
cause_counter = Counter(r['cause'] for r in diag_results)
print('\n' + '='*60)
print('원인별 FN 집계:')
for cause, cnt in cause_counter.most_common():
    print(f'  {cause:25s}: {cnt}건')

top_cause = cause_counter.most_common(1)[0][0] if cause_counter else None
print(f'\n→ 가장 많은 원인: [{top_cause}] → 다음 셀에서 수정 방향 제시')

In [ ]:
import pandas as pd

# 상세 테이블 출력
if diag_results:
    df_diag = pd.DataFrame(diag_results)
    display(df_diag[['video','gt','cause','max_persons','gate_det_f',
                      'max_jump_score','min_AR','max_streak','unique_ids','detail']])

# 원인별 자동 수정 가이드 출력
print('\n' + '='*60)
print(f'[{top_cause}] 수정 가이드:')

if top_cause == 'person_detection':
    print('  → conf_threshold 낮추기: YOLODetector(conf_threshold=0.3)')
    print('  → 또는 더 큰 모델로 교체: yolo11s.pt')

elif top_cause == 'gate_detection':
    print('  → gate_best.pt conf 낮추기: GateDetector(conf_thres=0.15)')
    print('  → gate 미탐지 영상에서 gate가 화면에 보이는지 먼저 확인 필요')
    print('  → 보인다면 gate_best.pt 재학습 (라벨 추가) 고려')

elif top_cause == 'gate_area_filter':
    print('  → near_gate_x margin_ratio 올리기: 0.2 → 0.4')
    print('  → run_pipeline() 내 near_gate_x 호출부 수정')

elif top_cause == 'rule_threshold':
    # 어떤 클래스가 많은지 세부 분류
    rule_fn = [r for r in diag_results if r['cause'] == 'rule_threshold']
    gt_cnt  = Counter(r['gt'] for r in rule_fn)
    print(f'  세부: {dict(gt_cnt)}')
    if gt_cnt.get('jump', 0) >= max(gt_cnt.values()):
        scores = [r['max_jump_score'] for r in rule_fn if r['gt'] == 'jump']
        print(f'  jump score 분포: min={min(scores):.3f} mean={sum(scores)/len(scores):.3f} max={max(scores):.3f}')
        print('  → dynamic_multiplier 낮추기: 5.0 → 3.0')
        print('  → 또는 min_score 낮추기: 0.25 → 0.20')
    if gt_cnt.get('crawling', 0) >= max(gt_cnt.values()):
        ars = [r['min_AR'] for r in rule_fn if r['gt'] == 'crawling']
        print(f'  crawling min_AR 분포: min={min(ars):.2f} mean={sum(ars)/len(ars):.2f} max={max(ars):.2f}')
        print('  → crawl_aspect_ratio_thresh 올리기: 1.6 → 1.8')
    if gt_cnt.get('tailgating', 0) >= max(gt_cnt.values()):
        streaks = [r['max_streak'] for r in rule_fn if r['gt'] == 'tailgating']
        print(f'  tailgating max_streak 분포: min={min(streaks)} mean={sum(streaks)/len(streaks):.1f}')
        print('  → min_cooccupy_frames 낮추기: 5 → 3')

elif top_cause == 'id_switch_suspected':
    print('  → ByteTrack lost_track_buffer 늘리기: sv.ByteTrack(lost_track_buffer=60)')
    print('  → IDRemapper frame_thresh 늘리기: IDRemapper(frame_thresh=30)')

# CSV 저장
csv_path = f'{DRIVE_SAVE}/diag_fn.csv'
os.makedirs(DRIVE_SAVE, exist_ok=True)
if diag_results:
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=diag_results[0].keys())
        writer.writeheader()
        writer.writerows(diag_results)
    print(f'\nCSV 저장 → {csv_path}')

### 10-B. 수정 적용 후 전체 재실행 & 비교

위 가이드 보고 파라미터 수정한 뒤 이 셀 실행 → before/after 비교표 출력

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Fix2+3: 여기서 파라미터 수정 → run_pipeline에 전부 전달됩니다
# ══════════════════════════════════════════════════════════════════
CONF_THRESHOLD        = 0.4
GATE_CONF             = 0.25
NEAR_GATE_MARGIN      = 0.2
JUMP_MULTIPLIER       = 5.0
JUMP_MIN_SCORE        = 0.25
JUMP_MIN_FRAMES       = 3
CRAWL_AR_THRESH       = 1.6
CRAWL_MIN_FRAMES      = 10
TAILGATE_MIN_FRAMES   = 5
GATE_OVERLAP_THRESH   = 0.25
TAILGATE_MAX_DIST     = 200
ID_REMAP_FRAME_THRESH = 15

# Fix3: 모든 파라미터를 CONFIG_DICT_V2에 반영
CONFIG_DICT_V2 = dict(CONFIG_DICT)
CONFIG_DICT_V2['crawl_aspect_ratio_thresh'] = CRAWL_AR_THRESH
CONFIG_DICT_V2['crawl_min_frames']          = CRAWL_MIN_FRAMES
CONFIG_DICT_V2['jump_min_frames']           = JUMP_MIN_FRAMES
CONFIG_DICT_V2['tailgating_min_frames']     = TAILGATE_MIN_FRAMES
CONFIG_DICT_V2['gate_overlap_thresh']       = GATE_OVERLAP_THRESH
CONFIG_DICT_V2['tailgate_max_dist']         = float(TAILGATE_MAX_DIST)
cfg_v2 = GateZoneConfig.from_dict(CONFIG_DICT_V2)

print('[CONFIG_V2] 적용값 확인')
for k in ['crawl_aspect_ratio_thresh','crawl_min_frames','jump_min_frames',
          'tailgating_min_frames','gate_overlap_thresh','tailgate_max_dist']:
    print(f'  {k:30s} = {getattr(cfg_v2, k)}')

# ── 전체 재실행 ───────────────────────────────────────────────────
all_candidates_v2 = []
video_event_map_v2 = {}

for vp in tqdm_nb(target_videos, desc='재실행'):
    cands_i, _, _ = run_pipeline(          # Fix2: 3-tuple + 모든 파라미터 전달
        vp, cfg_v2,
        output_dir=OUTPUT_DIR + '/v2',
        conf=CONF_THRESHOLD,
        gate_conf=GATE_CONF,
        near_gate_margin=NEAR_GATE_MARGIN,
        jump_multiplier=JUMP_MULTIPLIER,
        jump_min_score=JUMP_MIN_SCORE,
        id_remap_frame_thresh=ID_REMAP_FRAME_THRESH,
    )
    all_candidates_v2.extend(cands_i)
    video_event_map_v2[os.path.basename(vp)] = {c.event_type for c in cands_i}

# ── Before / After 비교 ───────────────────────────────────────────
def score_map(vmap, target_videos, event_classes=EVENT_CLASSES):
    tp = tn = fp = fn = 0
    for vp in target_videos:
        gt  = get_gt_from_path(vp)   # Fix1: 폴더명 기준
        det = vmap.get(os.path.basename(vp), set())
        if gt == 'normal':
            if not det: tn += 1
            else:       fp += 1
        else:
            if gt in det: tp += 1
            else:         fn += 1
    total = tp + tn + fp + fn
    acc       = round((tp+tn)/total*100, 1) if total else 0
    recall    = round(tp/(tp+fn)*100, 1)    if (tp+fn) else 0
    precision = round(tp/(tp+fp)*100, 1)    if (tp+fp) else 0
    return dict(TP=tp, TN=tn, FP=fp, FN=fn, Acc=f'{acc}%',
                Recall=f'{recall}%', Precision=f'{precision}%')

before = score_map(video_event_map,    target_videos)
after  = score_map(video_event_map_v2, target_videos)

print(f'{"":15s} {"Before":>10} {"After":>10}')
print('-' * 37)
for k in ['TP','TN','FP','FN','Acc','Recall','Precision']:
    b, a = str(before[k]), str(after[k])
    try:
        bv = float(b.rstrip('%')); av = float(a.rstrip('%'))
        better_high = k in ('TP','TN','Acc','Recall','Precision')
        mark = ' ✅' if (better_high and av>bv) else ' ❌' if (better_high and av<bv) else \
               ' ✅' if (k in ('FP','FN') and av<bv) else ' ❌' if (k in ('FP','FN') and av>bv) else ''
    except: mark = ''
    print(f'{k:15s} {b:>10} {a:>10}{mark}')

# ── 결과 저장 ─────────────────────────────────────────────────────
v2_settings = {
    'conf_threshold'        : CONF_THRESHOLD,
    'gate_conf'             : GATE_CONF,
    'near_gate_margin'      : NEAR_GATE_MARGIN,
    'jump_multiplier'       : JUMP_MULTIPLIER,
    'jump_min_score'        : JUMP_MIN_SCORE,
    'jump_min_frames'       : JUMP_MIN_FRAMES,
    'crawl_ar_thresh'       : CRAWL_AR_THRESH,
    'crawl_min_frames'      : CRAWL_MIN_FRAMES,
    'tailgate_min_frames'   : TAILGATE_MIN_FRAMES,
    'gate_overlap_thresh'   : GATE_OVERLAP_THRESH,
    'tailgate_max_dist'     : TAILGATE_MAX_DIST,
    'id_remap_frame_thresh' : ID_REMAP_FRAME_THRESH,
}
save_run_summary(
    label='tuned_v2',
    candidates=all_candidates_v2,
    vmap=video_event_map_v2,
    target_videos=target_videos,
    settings=v2_settings,
    out_dir=OUTPUT_DIR + '/v2',
)
import shutil
os.makedirs(DRIVE_SAVE, exist_ok=True)
shutil.copy(OUTPUT_DIR + '/v2/run_log.json', f'{DRIVE_SAVE}/run_log_tuned.json')
print(f'Drive 저장 → {DRIVE_SAVE}/run_log_tuned.json')


In [ ]:
import glob
from IPython.display import Video, display

clips = sorted(glob.glob(f'{OUTPUT_DIR}/clips/*.mp4'))
print(f'클립 {len(clips)}개')
for p in clips[:3]:
    print(os.path.basename(p))
    display(Video(p, embed=True, width=480))

## 10. Drive 저장

In [ ]:
if os.path.exists(DRIVE_SAVE):
    shutil.rmtree(DRIVE_SAVE)
shutil.copytree(OUTPUT_DIR, DRIVE_SAVE)
print(f'Drive 저장 완료 → {DRIVE_SAVE}')

## 11. 튜닝 가이드

### jump 안 잡힐 때
- 섹션 8-A의 score 값 확인
- score < 0.30이면: `jump_ratio=0.20` 또는 `dynamic_multiplier=2.0` 낮추기

### crawling 안 잡힐 때
- 섹션 8-B 실행해서 `in_crawl`, `ar` 값 확인
- `in_crawl=False`: gate 탐지 후 crawl_zone이 실제 영상 위치와 다름 → 섹션 5 미리보기에서 CRAWL(auto) 박스 위치 확인
- `ar` 값이 1.5보다 크면: `crawl_aspect_ratio_thresh=2.0`으로 올리기
- `crawl_min_frames=5`로 낮춰보기

### tailgating 오탐(FP) 많을 때 — ID switch로 인한 오탐
- `min_cooccupy_frames` 올리기 (기본 8 → 12~15)
  - ID switch는 ByteTrack이 보통 1~3프레임 안에 재할당하므로 streak이 길면 걸러짐

### tailgating 미탐(FN) 많을 때
- `min_cooccupy_frames` 내리기 (8 → 4~5)
  - gate 통과 시간이 짧아 streak을 못 채울 경우

### gate 위치 필터가 너무 좁을 때 (jump FN)
- `near_gate_x()` `margin_ratio=0.2` → `0.4`로 올리기

조정 후 섹션 4-D(Rules) → 섹션 6 → 섹션 7 재실행